In [2]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
)

In [9]:
message_test = """
-第一轮
    ("user", "你好，今天北京天气怎么样？"),
    ("assistant", "北京今天晴天，25°C，适合出行。需要我推荐景点吗？"),
-第二轮
    ("user", "推荐几个编程学习资源"),
    ("assistant", "推荐：1. GitHub开源项目 2. LeetCode刷题 3. MDN文档"),
-第三轮
    ("user", "能帮我写一段Python代码吗？实现快速排序"),
    ("assistant", "```python\ndef quicksort(arr):\n    if len(arr) <= 1: return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)\n```"),
-第四轮
    ("user", "这段代码的时间复杂度是多少？"),
    ("assistant", "快速排序平均时间复杂度O(n log n)，最坏情况O(n²)"),
-第五轮
    ("user", "如何优化最坏情况？"),
    ("assistant", "可以使用随机化pivot选择，或改用三路快排处理重复元素"),
"""

case_prompt = """
case 1
用户没有提供参数 默认N为3轮，3轮之后开启进项对 将3轮之前的对话进行总结摘要，并删除3轮之前的对话
case 2
用户提交非法参数就是非数字类型参数 ，给一个提示 但是也3轮之后开启进项对 将3轮之前的对话进行总结摘要，并删除3轮之前的对话
case 3
用户提交<0的参数 给一个提示 但是也3轮之后开启进项对 将3轮之前的对话进行总结摘要，并删除3轮之前的对话
case 4
用户提交 >0 但是 <3 的参数 ,给一个提示，保留3轮对话
case 5
用户提交>3的参数 ，根据用户提交的参数 开启对应N轮之前的的对话进行总结摘要，并删除3轮之前的对话
"""
prompt = f"""
请按照我的需求帮我实现 openai编程多轮对话的管理上下文的代码
这里是需求
-这是一个基于摘要压缩法：长对话时生成摘要替代历史
-需要一个参数来控制N轮对话之后开启生成摘要,然后删除N轮之前的对话
-每轮对话需要包含 用户的content 和 模型返回的 content

只需要给我最后的代码是需要使用openai的api方式可以直接运行的 并且需要包含 对应的运行方式 其他内容不允许输出，你需要验证通过之后再给我代码部分
验证逻辑如下
这里是模拟的用户和 大模型的N轮对话
{message_test}

以下是case
{case_prompt}
"""
messages = [{"role": "user", "content": prompt}]

response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=messages
)
print(response.choices[0].message.content)

```python
import openai
from typing import List, Dict, Union, Optional, Tuple

class OpenAIConversationManager:
    def __init__(self, api_key: str, summary_threshold: Union[int, str, None] = None):
        """
        初始化对话管理器
        :param api_key: OpenAI API密钥
        :param summary_threshold: N轮对话后开启摘要，None表示使用默认值3
        """
        self.client = openai.OpenAI(api_key=api_key)
        self.history: List[Dict[str, str]] = []
        self.summary: str = ""
        self.summary_threshold = self._validate_threshold(summary_threshold)
    
    def _validate_threshold(self, threshold) -> int:
        """
        验证参数逻辑：
        - Case 1: None -> 默认3
        - Case 2: 非数字 -> 提示，默认3
        - Case 3: <0 -> 提示，默认3
        - Case 4: 0<阈值<3 -> 提示，保留3
        - Case 5: >=3 -> 使用用户值
        """
        if threshold is None:
            return 3
        
        try:
            n = int(threshold)
            if n < 0:
                print(f"[提示] 参数{n}为负数，使用默认值3")
                return 3
  

In [8]:
import openai
from typing import List, Dict, Optional, Union


class ConversationManager:
    def __init__(self, client, max_history: Optional[Union[int, str]] = None):
        self.client = client
        self.max_history = self._validate_param(max_history)
        self.messages: List[Dict[str, str]] = []
        self.current_round = 0

    def _validate_param(self, param) -> int:
        default_n = 3

        if param is None:
            return default_n

        try:
            val = int(param)
            if val < 0:
                print(f"[系统提示] 参数不能小于0，已使用默认值 {default_n}")
                return default_n
            elif 0 < val < 3:
                print(f"[系统提示] 参数至少为3，已使用默认值 {default_n}")
                return default_n
            elif val >= 3:
                return val
            else:  # val == 0
                print(f"[系统提示] 参数至少为3，已使用默认值 {default_n}")
                return default_n
        except (ValueError, TypeError):
            print(f"[系统提示] 非法参数，已使用默认值 {default_n}")
            return default_n

    def add_message(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
        if role == "assistant":
            self.current_round += 1
            self._compress_if_needed()

    def _compress_if_needed(self):
        if self.current_round > self.max_history:
            # 分离system消息和对话消息
            system_msgs = [m for m in self.messages if m["role"] == "system"]
            dialogues = [m for m in self.messages if m["role"] != "system"]

            # 计算需要压缩的消息数量 (current_round - max_history)轮 * 2条/轮
            compress_rounds = self.current_round - self.max_history
            compress_msg_count = compress_rounds * 2

            if len(dialogues) > compress_msg_count:
                to_compress = dialogues[:compress_msg_count]
                to_keep = dialogues[compress_msg_count:]

                # 生成摘要
                summary = self._generate_summary(to_compress)

                # 重建消息列表: system + 摘要 + 保留的最近N轮
                self.messages = system_msgs
                self.messages.append({
                    "role": "system",
                    "content": f"[历史摘要] 前{compress_rounds}轮对话总结: {summary}"
                })
                self.messages.extend(to_keep)

                print(f"[压缩执行] 已将前{compress_rounds}轮对话压缩为摘要，当前保留最近{self.max_history}轮详细对话")

    def _generate_summary(self, messages: List[Dict[str, str]]) -> str:
        try:
            summary_prompt = [
                {"role": "system", "content": "请对以下对话历史进行简洁的摘要总结，保留关键信息和上下文："},
                {"role": "user", "content": "\n".join([f"{m['role']}: {m['content'][:100]}" for m in messages])}
            ]

            response = self.client.chat.completions.create(
                model="kimi-k2.5",
                messages=summary_prompt,
            )
            return response.choices[0].message.content
        except Exception as e:
            print(e)
            return f"摘要生成失败，包含{len(messages) // 2}轮对话"

    def chat(self, user_content: str) -> str:
        self.add_message("user", user_content)

        try:
            response = self.client.chat.completions.create(
                model="kimi-k2.5",
                messages=self.messages,
            )
            assistant_content = response.choices[0].message.content
            self.add_message("assistant", assistant_content)
            return assistant_content
        except Exception as e:
            error_msg = f"API调用错误: {str(e)}"
            self.add_message("assistant", error_msg)
            return error_msg

    def get_history(self) -> List[Dict[str, str]]:
        return self.messages.copy()


def run_test_cases():
    # 模拟对话数据
    dialogues = [
        ("user", "你好，今天北京天气怎么样？"),
        ("assistant", "北京今天晴天，25°C，适合出行。需要我推荐景点吗？"),
        ("user", "推荐几个编程学习资源"),
        ("assistant", "推荐：1. GitHub开源项目 2. LeetCode刷题 3. MDN文档"),
        ("user", "能帮我写一段Python代码吗？实现快速排序"),
        ("assistant",
         "```python\ndef quicksort(arr):\n    if len(arr) <= 1: return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)\n```"),
        ("user", "这段代码的时间复杂度是多少？"),
        ("assistant", "快速排序平均时间复杂度O(n log n)，最坏情况O(n²)"),
        ("user", "如何优化最坏情况？"),
        ("assistant", "可以使用随机化pivot选择，或改用三路快排处理重复元素"),
    ]

    print("=" * 60)
    print("Case 1: 无参数（默认N=3）")
    print("=" * 60)
    mgr1 = ConversationManager(client=client)
    for i in range(0, min(10, len(dialogues)), 2):
        user_msg = dialogues[i][1]
        asst_msg = dialogues[i + 1][1] if i + 1 < len(dialogues) else "模拟回复"
        mgr1.add_message("user", user_msg)
        mgr1.add_message("assistant", asst_msg)
        print(f"第{(i // 2) + 1}轮后消息数: {len(mgr1.get_history())}")

    for message in mgr1.get_history():
        print(message)

    print("\n" + "=" * 60)
    print("Case 2: 非法参数（字符串'abc'）")
    print("=" * 60)
    mgr2 = ConversationManager(client=client, max_history="abc")
    print(f"实际使用的N值: {mgr2.max_history}")
    for message in mgr2.get_history():
        print(message)

    print("\n" + "=" * 60)
    print("Case 3: 参数<0（-5）")
    print("=" * 60)
    mgr3 = ConversationManager(client=client, max_history=-5)
    print(f"实际使用的N值: {mgr3.max_history}")
    for message in mgr3.get_history():
        print(message)
    print("\n" + "=" * 60)
    print("Case 4: 参数>0但<3（2）")
    print("=" * 60)
    mgr4 = ConversationManager(client=client, max_history=2)
    print(f"实际使用的N值: {mgr4.max_history}")
    for message in mgr4.get_history():
        print(message)
    print("\n" + "=" * 60)
    print("Case 5: 参数>3（5）")
    print("=" * 60)
    mgr5 = ConversationManager(client=client, max_history=5)
    # 模拟6轮对话观察压缩行为
    extended_dialogues = list(dialogues) + [
        ("user", "还有其他排序算法推荐吗？"),
        ("assistant", "还可以学习归并排序、堆排序，它们的时间复杂度都是O(n log n)"),
    ]
    for i in range(0, min(12, len(extended_dialogues)), 2):
        user_msg = extended_dialogues[i][1]
        asst_msg = extended_dialogues[i + 1][1] if i + 1 < len(extended_dialogues) else "模拟回复"
        mgr5.add_message("user", user_msg)
        mgr5.add_message("assistant", asst_msg)
        round_num = (i // 2) + 1
        print(f"第{round_num}轮后消息数: {len(mgr5.get_history())}")
        if round_num > 5:
            print(f"  -> 已触发压缩，当前历史包含摘要+最近{mgr5.max_history}轮")
    for message in mgr5.get_history():
        print(message)


if __name__ == "__main__":
    run_test_cases()

Case 1: 无参数（默认N=3）
第1轮后消息数: 2
第2轮后消息数: 4
第3轮后消息数: 6
[压缩执行] 已将前1轮对话压缩为摘要，当前保留最近3轮详细对话
第4轮后消息数: 7
[压缩执行] 已将前2轮对话压缩为摘要，当前保留最近3轮详细对话
第5轮后消息数: 6
{'role': 'system', 'content': '[历史摘要] 前1轮对话总结: 用户询问北京今日天气，助手回复为晴天、25°C、适宜出行，并询问是否需要推荐景点。'}
{'role': 'system', 'content': '[历史摘要] 前2轮对话总结: **摘要总结：**\n\n用户先后提出两个需求：1）询问编程学习资源推荐，助手建议了GitHub开源项目、LeetCode刷题和MDN文档；2）请求编写Python快速排序代码，助手提供了递归实现的代码片段（基于中间轴点分区）。'}
{'role': 'user', 'content': '这段代码的时间复杂度是多少？'}
{'role': 'assistant', 'content': '快速排序平均时间复杂度O(n log n)，最坏情况O(n²)'}
{'role': 'user', 'content': '如何优化最坏情况？'}
{'role': 'assistant', 'content': '可以使用随机化pivot选择，或改用三路快排处理重复元素'}

Case 2: 非法参数（字符串'abc'）
[系统提示] 非法参数，已使用默认值 3
实际使用的N值: 3

Case 3: 参数<0（-5）
[系统提示] 参数不能小于0，已使用默认值 3
实际使用的N值: 3

Case 4: 参数>0但<3（2）
[系统提示] 参数至少为3，已使用默认值 3
实际使用的N值: 3

Case 5: 参数>3（5）
第1轮后消息数: 2
第2轮后消息数: 4
第3轮后消息数: 6
第4轮后消息数: 8
第5轮后消息数: 10
[压缩执行] 已将前1轮对话压缩为摘要，当前保留最近5轮详细对话
第6轮后消息数: 11
  -> 已触发压缩，当前历史包含摘要+最近5轮
{'role': 'system', 'content': '[历史摘要] 前1轮对话总结: 用户询问北京今日天气状况，得知为晴天25°C且适宜出行，随后助手

In [17]:
import openai
from typing import List, Dict, Union, Optional, Tuple


class OpenAIConversationManager:
    def __init__(self, client, summary_threshold: Union[int, str, None] = None):
        """
        初始化对话管理器
        :param api_key: OpenAI API密钥
        :param summary_threshold: N轮对话后开启摘要，None表示使用默认值3
        """
        self.client = client
        self.history: List[Dict[str, str]] = []
        self.summary: str = ""
        self.summary_threshold = self._validate_threshold(summary_threshold)

    def _validate_threshold(self, threshold) -> int:
        """
        验证参数逻辑：
        - Case 1: None -> 默认3
        - Case 2: 非数字 -> 提示，默认3
        - Case 3: <0 -> 提示，默认3
        - Case 4: 0<阈值<3 -> 提示，保留3
        - Case 5: >=3 -> 使用用户值
        """
        if threshold is None:
            return 3

        try:
            n = int(threshold)
            if n < 0:
                print(f"[提示] 参数{n}为负数，使用默认值3")
                return 3
            elif 0 < n < 3:
                print(f"[提示] 参数{n}小于3，保留3轮对话")
                return 3
            elif n >= 3:
                return n
            else:  # n == 0
                print(f"[提示] 参数{n}无效，使用默认值3")
                return 3
        except (ValueError, TypeError):
            print(f"[提示] 非法参数'{threshold}'，使用默认值3")
            return 3

    def _generate_summary(self, messages: List[Dict[str, str]]) -> str:
        """生成对话摘要"""
        prompt = "请简要总结以下对话的关键信息（限制在100字内）：\n"
        for msg in messages:
            prompt += f"{msg['role']}: {msg['content']}\n"

        try:
            response = self.client.chat.completions.create(
                model="kimi-k2.5",
                messages=[
                    {"role": "system", "content": "你是一个对话摘要助手，请简明扼要地总结。"},
                    {"role": "user", "content": prompt}
                ],
            )
            return response.choices[0].message.content
        except Exception as e:
            # API失败时返回简单拼接
            return " | ".join([f"{m['role']}:{m['content'][:30]}..." for m in messages])

    def chat(self, user_message: str) -> str:
        """
        发送消息并管理上下文
        返回助手回复
        """
        # 构建消息列表：摘要(如果有) + 当前历史 + 新消息
        messages = []
        if self.summary:
            messages.append({
                "role": "system",
                "content": f"[历史摘要] {self.summary}"
            })
        messages.extend(self.history)
        messages.append({"role": "user", "content": user_message})

        # 调用API
        try:
            response = self.client.chat.completions.create(
                model="kimi-k2.5",
                messages=messages,
            )
            assistant_message = response.choices[0].message.content

            # 保存到历史
            self.history.append({"role": "user", "content": user_message})
            self.history.append({"role": "assistant", "content": assistant_message})

            # 检查是否达到N轮（每轮=2条消息），触发摘要压缩
            if len(self.history) >= self.summary_threshold * 2:
                self._compress_history()

            return assistant_message

        except Exception as e:
            return f"Error: {str(e)}"

    def _compress_history(self):
        """摘要压缩：将前N轮生成摘要，删除原对话"""
        # 前N轮消息（2*N条）
        to_summarize = self.history[:self.summary_threshold * 2]
        # 保留剩余对话
        self.history = self.history[self.summary_threshold * 2:]

        # 生成并合并摘要
        new_summary = self._generate_summary(to_summarize)
        if self.summary:
            self.summary += f" | {new_summary}"
        else:
            self.summary = new_summary

        print(f"[系统] 已生成摘要，删除前{self.summary_threshold}轮，保留{len(self.history) // 2}轮对话")

    def get_context_info(self) -> Dict:
        """获取当前上下文信息（调试用）"""
        return {
            "summary": self.summary,
            "history_length": len(self.history),
            "history_rounds": len(self.history) // 2,
            "threshold": self.summary_threshold
        }


def simulate_conversations(client, threshold: Union[int, str, None] = None):
    """
    模拟运行方式：使用提供的5轮对话数据测试
    """
    # 提供的测试数据
    conversations: List[Tuple[str, str]] = [
        ("user", "你好，今天北京天气怎么样？"),
        ("assistant", "北京今天晴天，25°C，适合出行。需要我推荐景点吗？"),
        ("user", "推荐几个编程学习资源"),
        ("assistant", "推荐：1. GitHub开源项目 2. LeetCode刷题 3. MDN文档"),
        ("user", "能帮我写一段Python代码吗？实现快速排序"),
        ("assistant",
         "```python\ndef quicksort(arr):\n    if len(arr) <= 1: return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)\n```"),
        ("user", "这段代码的时间复杂度是多少？"),
        ("assistant", "快速排序平均时间复杂度O(n log n)，最坏情况O(n²)"),
        ("user", "如何优化最坏情况？"),
        ("assistant", "可以使用随机化pivot选择，或改用三路快排处理重复元素"),
    ]

    manager = OpenAIConversationManager(client, summary_threshold=threshold)

    print(f"\n{'=' * 60}")
    print(f"测试参数: {threshold if threshold is not None else '默认(3)'}")
    print(f"实际N值: {manager.summary_threshold}")
    print(f"{'=' * 60}")

    # 模拟多轮对话（实际使用时调用manager.chat()）
    for i in range(0, len(conversations), 2):
        user_role, user_content = conversations[i]
        assistant_role, assistant_content = conversations[i + 1]

        print(f"\n第{i // 2 + 1}轮:")
        print(f"User: {user_content}")

        # 模拟添加到历史（实际使用时通过chat方法自动添加）
        manager.history.append({"role": "user", "content": user_content})
        manager.history.append({"role": "assistant", "content": assistant_content})

        print(f"Assistant: {assistant_content[:60]}...")

        # 检查是否触发压缩（每轮结束检查，除了最后一轮）
        if len(manager.history) >= manager.summary_threshold * 2 and i < len(conversations) - 2:
            manager._compress_history()
        for msg in manager.history:
            print(msg)

    # 最终结果
    info = manager.get_context_info()
    print(f"\n{'=' * 60}")
    print(f"对话结束 - 保留轮数: {info['history_rounds']}")
    print(f"摘要内容: {info['summary'][:100]}..." if info['summary'] else "无摘要")
    print(f"{'=' * 60}")


# 运行方式
if __name__ == "__main__":
    conversation = [
        "我叫张三，做后端开发3年了",
        "我想转行做AI工程师，有什么建议？",
        "需要学习哪些技术栈？详细说说",
        "推荐几个学习资源",
        "学习周期大概多久？",
        "我之前告诉你我叫什么？（测试摘要是否保留关键信息）",
        "还有我是做什么的？",
    ]

    max_content = 4
    manager = OpenAIConversationManager(client, summary_threshold=max_content)
    for msg in conversation:
        print('=' * 20)
        response = manager.chat(msg)
        # print(response)
        for msg in manager.history:
            print(msg)
        print('=' * 20)

{'role': 'user', 'content': '我叫张三，做后端开发3年了'}
{'role': 'assistant', 'content': '你好张三！👋 \n\n3年后端经验是个**挺关键的阶段**——既脱离了初级开发的"野蛮生长"期，又刚好站在**技术深度**和**职业方向**选择的分水岭上。\n\n不知道你目前更想聊哪方面？我可以针对性给建议：\n\n**技术进阶方向**\n- 系统架构设计（分布式、微服务、高并发）\n- 性能调优深度（JVM/数据库/缓存/网络）\n- 云原生技术栈（K8s、Service Mesh、可观测性）\n- 或者你当前技术栈的源码级深入（Java/Go/Python等）\n\n**职业发展困惑**\n- 继续深耕技术（P6/P7路线）vs 转向技术管理\n- 当前公司晋升瓶颈，考虑跳槽时机和准备\n- 35岁焦虑的提前布局（3年正是打基础的时候）\n\n**具体场景**\n- 正在准备面试（简历优化、八股文深度、项目亮点提炼）\n- 接手新项目的技术选型\n- 带新人/小团队的管理初体验\n\n**你现在用的是什么技术栈？最近最困扰你的是技术问题还是职业选择？** 先说说看，咱们具体聊。'}
{'role': 'user', 'content': '我叫张三，做后端开发3年了'}
{'role': 'assistant', 'content': '你好张三！👋 \n\n3年后端经验是个**挺关键的阶段**——既脱离了初级开发的"野蛮生长"期，又刚好站在**技术深度**和**职业方向**选择的分水岭上。\n\n不知道你目前更想聊哪方面？我可以针对性给建议：\n\n**技术进阶方向**\n- 系统架构设计（分布式、微服务、高并发）\n- 性能调优深度（JVM/数据库/缓存/网络）\n- 云原生技术栈（K8s、Service Mesh、可观测性）\n- 或者你当前技术栈的源码级深入（Java/Go/Python等）\n\n**职业发展困惑**\n- 继续深耕技术（P6/P7路线）vs 转向技术管理\n- 当前公司晋升瓶颈，考虑跳槽时机和准备\n- 35岁焦虑的提前布局（3年正是打基础的时候）\n\n**具体场景**\n- 正在准备面试（简历优化、八股文深度、项目亮点提炼）\n- 接手新项目的技术选型\n- 带新人/小团队